# ADC2023 Five-Gas Hybrid Quantum Training Notebook

This notebook gives you one end-to-end place to train the local `ariel_quantum_regression` model that now lives inside `sbi_ariel_adc2023`.

It does four things:

1. Resolve the repo and ADC2023 dataset paths
2. Verify the notebook kernel has the required Python packages
3. Prepare cached train/validation/holdout/test splits from the raw Ariel files
4. Run the hybrid quantum regressor training pipeline and preview outputs

This notebook is intentionally aligned with `ariel_quantum_regression/dataset.py`, `ariel_quantum_regression/model.py`, and `ariel_quantum_regression/training.py`, because that local package is the self-contained training path currently present in the repo.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
import time
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "ariel").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


def find_ariel_root(repo_root: Path) -> Path:
    env_root = os.environ.get("ARIEL_DATA_ROOT")
    candidates: list[Path] = []
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())
    candidates.extend([repo_root / "ariel", Path.cwd().resolve()])
    for candidate in candidates:
        if (candidate / "TrainingData" / "AuxillaryTable.csv").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the ADC2023 data root. Set ARIEL_DATA_ROOT or place the dataset under <repo>/ariel."
    )


WORKSPACE_START = Path.cwd().resolve()
REPO_ROOT = find_repo_root(WORKSPACE_START)
ARIEL_ROOT = find_ariel_root(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "ariel"
NOTEBOOK_ROOT = PACKAGE_ROOT / "models" / "notebooks" / "sbi_ariel_adc2023"
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")

CONFIG = {
    "output_dir": REPO_ROOT / "outputs" / "ariel_quantum_regression" / RUN_STAMP,
    "prepared_cache_dir": "prepared_cache",
    "seed": 42,
    "batch_size": 64,
    "eval_batch_size": 128,
    "max_epochs": 30,
    "early_stop_patience": 6,
    "scheduler_patience": 2,
    "scheduler_factor": 0.5,
    "classical_lr": 2.0e-3,
    "quantum_lr": 8.0e-4,
    "weight_decay": 1.0e-4,
    "gradient_clip_norm": 5.0,
    "dropout": 0.1,
    "loss_name": "mse",
    "qnn_qubits": 8,
    "qnn_depth": 2,
    "qnn_init_scale": 0.1,
    "classical_only": False,
    "quantum_warmup_epochs": 5,
    "quantum_ramp_epochs": 4,
    "quantum_backbone_freeze_epochs": 0,
    "use_amp": True,
    "quantum_use_async": False,
    "log_every_batches": 20,
    "train_limit": None,
    "val_limit": None,
    "holdout_limit": None,
    "test_limit": None,
}

for candidate in (REPO_ROOT, PACKAGE_ROOT):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

print(f"Python: {platform.python_version()}")
print(f"Working directory: {WORKSPACE_START}")
print(f"Repo root: {REPO_ROOT}")
print(f"Ariel root: {ARIEL_ROOT}")
print(f"Package root: {PACKAGE_ROOT}")
print("Notebook config:")
pprint(CONFIG)


In [ ]:
import importlib

REQUIRED_RAW_FILES = [
    ARIEL_ROOT / "TrainingData" / "AuxillaryTable.csv",
    ARIEL_ROOT / "TrainingData" / "Ground Truth Package" / "FM_Parameter_Table.csv",
    ARIEL_ROOT / "TrainingData" / "SpectralData.hdf5",
    ARIEL_ROOT / "TestData" / "AuxillaryTable.csv",
    ARIEL_ROOT / "TestData" / "SpectralData.hdf5",
]

THIRD_PARTY_MODULES = [
    "h5py",
    "numpy",
    "pandas",
    "sklearn",
    "torch",
]
if not CONFIG["classical_only"]:
    THIRD_PARTY_MODULES.extend(["pennylane"])

LOCAL_MODULES = [
    "models.notebooks.sbi_ariel_adc2023.ariel_quantum_regression.constants",
    "models.notebooks.sbi_ariel_adc2023.ariel_quantum_regression.dataset",
    "models.notebooks.sbi_ariel_adc2023.ariel_quantum_regression.model",
    "models.notebooks.sbi_ariel_adc2023.ariel_quantum_regression.training",
]


def module_status(module_name: str) -> dict[str, object]:
    try:
        importlib.import_module(module_name)
        return {"ok": True, "detail": "imported"}
    except Exception as exc:
        return {"ok": False, "detail": f"{type(exc).__name__}: {exc}"}


file_status = {str(path.relative_to(ARIEL_ROOT)): path.exists() for path in REQUIRED_RAW_FILES}
third_party_status = {name: module_status(name) for name in THIRD_PARTY_MODULES}
local_status = {name: module_status(name) for name in LOCAL_MODULES}

PRECHECK = {
    "files": file_status,
    "third_party_modules": third_party_status,
    "local_modules": local_status,
}

print(json.dumps(PRECHECK, indent=2))

missing_files = [name for name, ok in file_status.items() if not ok]
missing_third_party = [name for name, payload in third_party_status.items() if not payload["ok"]]
missing_local = [name for name, payload in local_status.items() if not payload["ok"]]
PRECHECK_OK = not (missing_files or missing_third_party or missing_local)

print()
if PRECHECK_OK:
    print("Preflight passed. You can run the workflow cells below.")
else:
    print("Preflight failed. Fix the reported issues before training.")
    if missing_files:
        print("Missing raw dataset files:")
        for name in missing_files:
            print(f"- {name}")
    if missing_third_party:
        print("Missing third-party packages in this kernel:")
        for name in missing_third_party:
            print(f"- {name}: {third_party_status[name]['detail']}")
    if missing_local:
        print("Missing local training modules:")
        for name in missing_local:
            print(f"- {name}: {local_status[name]['detail']}")


## Preflight Notes

If `PRECHECK_OK` is `False`, fix the missing pieces before running the workflow cells.

Expected third-party packages for hybrid mode:

- `h5py`
- `numpy`
- `pandas`
- `scikit-learn`
- `torch`
- `pennylane`
- `pennylane-lightning` or `pennylane-lightning-gpu`

If you want a quick classical sanity run, set `CONFIG["classical_only"] = True` in the first code cell and rerun preflight.

Example install command for the active kernel:

```python
%pip install h5py numpy pandas scikit-learn torch pennylane pennylane-lightning
```

This notebook now targets the local package under `models.notebooks.sbi_ariel_adc2023.ariel_quantum_regression`, not the older FMPE wrapper files.

In [ ]:
if not PRECHECK_OK:
    raise RuntimeError("Fix the preflight issues above before running the workflow cells.")

import pandas as pd
from IPython.display import display

from models.notebooks.sbi_ariel_adc2023.ariel_quantum_regression.constants import AUX_COLUMNS, TARGET_COLUMNS
from models.notebooks.sbi_ariel_adc2023.ariel_quantum_regression.dataset import prepare_data
from models.notebooks.sbi_ariel_adc2023.ariel_quantum_regression.training import TrainingConfig, run_training_experiment

cfg = TrainingConfig(
    project_root=str(REPO_ROOT),
    data_root=str(ARIEL_ROOT),
    output_dir=str(CONFIG["output_dir"]),
    prepared_cache_dir=CONFIG["prepared_cache_dir"],
    seed=int(CONFIG["seed"]),
    batch_size=int(CONFIG["batch_size"]),
    eval_batch_size=int(CONFIG["eval_batch_size"]),
    max_epochs=int(CONFIG["max_epochs"]),
    early_stop_patience=int(CONFIG["early_stop_patience"]),
    scheduler_patience=int(CONFIG["scheduler_patience"]),
    scheduler_factor=float(CONFIG["scheduler_factor"]),
    classical_lr=float(CONFIG["classical_lr"]),
    quantum_lr=float(CONFIG["quantum_lr"]),
    weight_decay=float(CONFIG["weight_decay"]),
    gradient_clip_norm=float(CONFIG["gradient_clip_norm"]),
    dropout=float(CONFIG["dropout"]),
    loss_name=str(CONFIG["loss_name"]),
    qnn_qubits=int(CONFIG["qnn_qubits"]),
    qnn_depth=int(CONFIG["qnn_depth"]),
    qnn_init_scale=float(CONFIG["qnn_init_scale"]),
    classical_only=bool(CONFIG["classical_only"]),
    quantum_warmup_epochs=int(CONFIG["quantum_warmup_epochs"]),
    quantum_ramp_epochs=int(CONFIG["quantum_ramp_epochs"]),
    quantum_backbone_freeze_epochs=int(CONFIG["quantum_backbone_freeze_epochs"]),
    use_amp=bool(CONFIG["use_amp"]),
    quantum_use_async=bool(CONFIG["quantum_use_async"]),
    log_every_batches=int(CONFIG["log_every_batches"]),
    train_limit=CONFIG["train_limit"],
    val_limit=CONFIG["val_limit"],
    holdout_limit=CONFIG["holdout_limit"],
    test_limit=CONFIG["test_limit"],
)

display(
    pd.DataFrame(
        [{"item": key, "value": value} for key, value in cfg.to_json_dict().items() if key not in {"aux_columns", "target_columns"}]
    )
)
print(f"Aux columns: {AUX_COLUMNS}")
print(f"Target columns: {TARGET_COLUMNS}")


In [ ]:
prepared = prepare_data(
    data_root=cfg.resolved_data_root(),
    output_dir=cfg.resolved_output_dir(),
    prepared_cache_dir=cfg.resolved_prepared_cache_dir(),
    seed=cfg.seed,
    train_limit=cfg.train_limit,
    val_limit=cfg.val_limit,
    holdout_limit=cfg.holdout_limit,
    test_limit=cfg.test_limit,
)

print(json.dumps(prepared.prepared_manifest, indent=2, sort_keys=True))
print(json.dumps(prepared.split_manifest, indent=2, sort_keys=True))

split_rows = [
    {
        "split": "train",
        "rows": prepared.train.rows,
        "aux_shape": tuple(prepared.train.aux.shape),
        "spectra_shape": tuple(prepared.train.spectra.shape),
        "targets_shape": tuple(prepared.train.targets.shape),
    },
    {
        "split": "val",
        "rows": prepared.val.rows,
        "aux_shape": tuple(prepared.val.aux.shape),
        "spectra_shape": tuple(prepared.val.spectra.shape),
        "targets_shape": tuple(prepared.val.targets.shape),
    },
    {
        "split": "holdout",
        "rows": prepared.holdout.rows,
        "aux_shape": tuple(prepared.holdout.aux.shape),
        "spectra_shape": tuple(prepared.holdout.spectra.shape),
        "targets_shape": tuple(prepared.holdout.targets.shape),
    },
    {
        "split": "testdata",
        "rows": prepared.testdata.rows,
        "aux_shape": tuple(prepared.testdata.aux.shape),
        "spectra_shape": tuple(prepared.testdata.spectra.shape),
        "targets_shape": None,
    },
]

display(pd.DataFrame(split_rows))
print(f"Wavelength bins: {len(prepared.wavelength_um)}")
print(f"Train sample IDs: {prepared.train.planet_ids[:5].tolist()}")


In [ ]:
results = run_training_experiment(cfg)
summary = results["summary"]
output_dir = Path(summary["output_dir"])

print(json.dumps(summary, indent=2, sort_keys=True))

metrics_rows = [
    {
        "split": "validation",
        "rmse_mean": float(results["validation_metrics"]["rmse_mean"]),
        "mae_mean": float(results["validation_metrics"]["mae_mean"]),
    },
    {
        "split": "holdout",
        "rmse_mean": float(results["holdout_metrics"]["rmse_mean"]),
        "mae_mean": float(results["holdout_metrics"]["mae_mean"]),
    },
]
display(pd.DataFrame(metrics_rows))

history_path = output_dir / "history.csv"
if history_path.exists():
    display(pd.read_csv(history_path).tail(10))
else:
    print("No history.csv found in the output directory.")

for artifact_name in (
    "validation_metrics.json",
    "holdout_metrics.json",
    "run_summary.json",
):
    artifact_path = output_dir / artifact_name
    if artifact_path.exists():
        print(f"\n=== {artifact_name} ===")
        print(artifact_path.read_text())

test_predictions_path = output_dir / "testdata_predictions.csv"
if test_predictions_path.exists():
    display(pd.read_csv(test_predictions_path).head())

artifact_rows = [
    {
        "path": str(path.relative_to(REPO_ROOT)),
        "size_kb": round(path.stat().st_size / 1024.0, 1),
    }
    for path in sorted(output_dir.iterdir())
    if path.is_file()
]
display(pd.DataFrame(artifact_rows))
